# Lesson 2 — The agent loop

**You will:** wrap an LLM call in a loop, give it a tool, and watch the four-move agent loop happen turn by turn.

[Open in Colab](https://colab.research.google.com/github/richey-malhotra/barebear/blob/main/lessons/02-the-agent-loop/lesson.ipynb)
[Read the lesson narrative](./lesson.md)
[Back to syllabus](../README.md)

## The big idea

In Lesson 1 you sent one message to an LLM and got one reply. That's a chat completion. An **agent** is what you get when you wrap that call in a loop and let the model *act* on the world between turns.

Every agent loop, in every framework, has the same four moves: **perceive → think → act → observe**. The loop ends when the model returns plain text instead of asking for another tool call. That's it. That's the whole secret.

## Setup

Run the next two cells once per session. They install barebear and set your OpenRouter key (free tier — get one at <https://openrouter.ai>).

In [ ]:
%pip install -q "barebear[openai]"

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Paste your OpenRouter key (sk-or-...): ")
print("Key set. First 8 characters:", os.environ["OPENROUTER_API_KEY"][:8] + "...")

## Your first agent

A one-tool agent. You give it `greet(name)`, then ask it to greet a user. Pass `trace=True` so you can watch the loop happen.

In [ ]:
from barebear import Bear, Task, Tool, OpenRouterModel

def greet(name: str) -> str:
    return f"Hello, {name}!"

bear = Bear(
    model=OpenRouterModel(),
    tools=[Tool(name="greet", fn=greet, description="Greet someone by name")],
)

report = bear.run(Task(goal="Greet a user named Alice"), trace=True)
print(report.summary())

## What just happened

You should have seen something like:

```
─── turn 1 ───
> calling model with 2 messages, 1 tools available
< tool call: greet(name="Alice")
→ greet returned: Hello, Alice!

─── turn 2 ───
< final answer: Hello, Alice! How can I help you today?
```

Two turns. **Turn 1**: the model picked the tool. The framework ran the function. The result went back in the messages. **Turn 2**: with the tool result in context, the model decided no more tool calls were needed and produced a final answer.

That's the agent loop. Everything fancy in agentic AI is variations on this.

## Exercise

Add a second tool — `farewell(name)` — and run the agent with the goal *"Greet Alice and then say goodbye to her."* How many turns does it take?

In [ ]:
def farewell(name: str) -> str:
    return f"Goodbye, {name}!"

# Build a new bear with both tools and run it.

## Read the source

The agent loop is about 30 lines of Python. Open [`src/barebear/bear.py`](https://github.com/richey-malhotra/barebear/blob/main/src/barebear/bear.py) and find the `_run_loop` method. Trace the four moves directly in the code.

## What's next

[Lesson 3 →](../03-tool-design/lesson.ipynb)